# Experiment: Part C initial results

Objective:
- Train masked PPO and masked double dueling DQN on the LVHM environment.
- Compare them with the six dispatch rules on the same test shifts under the four Part B test conditions.
- Done when the results tables are saved under `results/`.

Run the cells from top to bottom. Training (sections 4 and 5) is the only slow part, and checkpoints that already exist are reused when the notebook is run again.


## 0. Setup

**Amazon SageMaker (T4 GPU)**
1. Start a notebook instance or Studio space with a T4 GPU (for example `ml.g4dn.xlarge`) and a Python 3.12 kernel.
2. In a terminal, clone the private repo: `git clone git@github.com:uts-amazing-abundance/lvhm.git` (SSH key or personal access token).
3. Open `lvhm/notebooks/experiments.ipynb`, run the install cell below once, then restart the kernel.

**Locally**: `uv sync --extra notebook`, then `uv run --extra notebook --with jupyterlab jupyter lab notebooks/experiments.ipynb`, and skip the install cell.


In [ ]:
# Install this repo plus pandas and matplotlib into the kernel. Run once, then restart the kernel.
# A CUDA build of torch that is already installed is kept (the package only needs torch>=2.2).
%pip install -q -e '..[notebook]'


## 1. Imports and device check


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch

from lvhm.agents.dqn import DQNConfig, load_dqn, run_dqn
from lvhm.agents.ppo import load_ppo, train_ppo
from lvhm.baselines import rule_baselines
from lvhm.config import PRODUCTS, EnvConfig
from lvhm.experiments import conditions, evaluate_conditions

# 'cuda' on the SageMaker T4, 'cpu' anywhere without a GPU
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE, torch.cuda.get_device_name(0) if DEVICE == 'cuda' else '')


## 2. Settings

Every setting lives in this cell. Each algorithm trains once per seed for the given number of environment steps.


In [ ]:
# Repo root, whether the kernel starts in notebooks/ or in the repo itself
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
RUN_DIR = ROOT / 'runs'         # checkpoints and learning curves
RESULTS_DIR = ROOT / 'results'  # tables and figures for the report
RESULTS_DIR.mkdir(exist_ok=True)

SEEDS = [0, 1, 2]          # one training run per seed
PPO_TIMESTEPS = 300_000    # Part B plans 1M steps; reduced for the initial results
DQN_TIMESTEPS = 300_000
N_TEST_SHIFTS = 100        # fixed test shifts per condition
RETRAIN = False            # False: skip training when a checkpoint already exists

ENV = EnvConfig()          # environment defaults, documented in the README


## 3. The environment at a glance

The agent never sees product names, only weight and fragility. Processing time and drop probability follow from those two features.


In [ ]:
# P1-P4 are familiar; P5 and P6 are never seen in training
products = pd.DataFrame(
    [
        {
            'product': name,
            'weight_kg': weight,
            'fragility': fragility,
            'processing_s': round(float(ENV.proc_time(weight, fragility)), 2),
            'drop_prob': round(float(ENV.drop_probability(weight, fragility)), 3),
        }
        for name, (weight, fragility) in PRODUCTS.items()
    ]
)
products


In [ ]:
# The four Part B test conditions (test 2 is split into P5 and P6)
pd.DataFrame([vars(c) for c in conditions(ENV)])


## 4. Train masked PPO

sb3-contrib `MaskablePPO`, with the shared-weight job scorer as its policy network.
- Every 25k steps the policy is scored on 20 fixed validation shifts; the best checkpoint is kept as `runs/ppo/seed<N>/best_model.zip`.
- The learning curve is written to `validation.csv` next to it.
- SB3 may warn that PPO is meant for the CPU. The network is small, so the GPU gains little here; pass `device='cpu'` if it turns out slower.


In [ ]:
ppo_runs = []
for seed in SEEDS:
    out = RUN_DIR / 'ppo' / f'seed{seed}'
    if (out / 'best_model.zip').exists() and not RETRAIN:
        print(f'PPO seed {seed}: checkpoint found, skipping training')
        continue
    # prints one validation line every 25k steps
    info = train_ppo(ENV, out, seed=seed, timesteps=PPO_TIMESTEPS, device=DEVICE)
    ppo_runs.append(info)
    print(info)


## 5. Train masked double dueling DQN

SB3's `DQN` has no action masking, no double-DQN target and no dueling head, so this agent is plain PyTorch (`lvhm.agents.dqn`). It shares the job scorer with PPO. The best checkpoint is kept as `runs/dqn/seed<N>/best_model.pt`.


In [ ]:
# DQNConfig holds every DQN hyperparameter (learning rate, buffer, epsilon schedule, ...)
dqn_config = DQNConfig(timesteps=DQN_TIMESTEPS, device=DEVICE)

dqn_runs = []
for seed in SEEDS:
    out = RUN_DIR / 'dqn' / f'seed{seed}'
    if (out / 'best_model.pt').exists() and not RETRAIN:
        print(f'DQN seed {seed}: checkpoint found, skipping training')
        continue
    info = run_dqn(ENV, out, seed=seed, dqn_config=dqn_config)
    dqn_runs.append(info)
    print({k: v for k, v in info.items() if k != 'config'})


## 6. Learning curves

Validation score against environment steps, one line per seed.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for ax, algo in zip(axes, ['ppo', 'dqn']):
    for seed in SEEDS:
        curve_file = RUN_DIR / algo / f'seed{seed}' / 'validation.csv'
        if curve_file.exists():
            curve = pd.read_csv(curve_file)
            ax.plot(curve['timesteps'], curve['score'], label=f'seed {seed}')
    ax.set_title(algo.upper())
    ax.set_xlabel('environment steps')
    ax.legend()
axes[0].set_ylabel('validation score')
fig.savefig(RESULTS_DIR / 'learning_curves.png', dpi=150, bbox_inches='tight')
plt.show()


## 7. Evaluate every policy on the test conditions

Rules and learned policies face the same 100 shifts in each condition, with the same drop outcomes (common random numbers). Each learned policy is evaluated once per seed, on the CPU.


In [ ]:
# A list means one policy trained with several seeds
policies = rule_baselines() | {
    'PPO': [load_ppo(RUN_DIR / 'ppo' / f'seed{s}' / 'best_model.zip') for s in SEEDS],
    'DQN': [load_dqn(RUN_DIR / 'dqn' / f'seed{s}' / 'best_model.pt', ENV) for s in SEEDS],
}

# One row per condition, policy, seed and shift
df = pd.DataFrame(evaluate_conditions(policies, ENV, n_shifts=N_TEST_SHIFTS))
df.to_csv(RESULTS_DIR / 'test_shifts.csv', index=False)
df.head()


## 8. Results tables for Part C

Mean ± standard deviation across shifts (and across seeds for the learned policies). Tardiness and travel are in seconds; `careful` counts careful-mode picks.


In [ ]:
METRICS = ['on_time', 'tardiness', 'travel', 'drops', 'scrapped', 'unfinished', 'careful', 'score']
ORDER = ['FCFS', 'SPT', 'LRF', 'EDF', 'NTF', 'Random', 'PPO', 'DQN']


def fmt(stats):
    # 'mean ± std' strings from a frame with 'mean' and 'std' columns
    return stats['mean'].map('{:.2f}'.format) + ' ± ' + stats['std'].map('{:.2f}'.format)


# Table 1: test 1 (normal day, familiar products), every metric; '-' marks a policy with no results
stats = df[df['condition'] == 'familiar'].groupby('policy')[METRICS].agg(['mean', 'std'])
familiar_table = pd.DataFrame({m: fmt(stats[m]) for m in METRICS}).reindex(ORDER).fillna('-')
familiar_table


In [ ]:
# Table 2: score under every test condition
stats = df.groupby(['policy', 'condition'])['score'].agg(['mean', 'std'])
score_table = fmt(stats).unstack('condition').reindex(index=ORDER, columns=[c.name for c in conditions(ENV)]).fillna('-')
score_table


In [ ]:
# Save both tables as Markdown to paste into the Part C document
def to_markdown(table):
    lines = ['| policy | ' + ' | '.join(table.columns) + ' |', '|---' * (len(table.columns) + 1) + '|']
    lines += ['| ' + name + ' | ' + ' | '.join(row) + ' |' for name, row in zip(table.index, table.values)]
    return '\n'.join(lines)


report = (
    '## Normal day, familiar products\n\n' + to_markdown(familiar_table)
    + '\n\n## Score by test condition\n\n' + to_markdown(score_table) + '\n'
)
(RESULTS_DIR / 'partc_tables.md').write_text(report)
print(report)


## 9. Changed penalties: does the policy adapt without retraining?

Test 4 raises the damage weight only through the observation. A policy that adapts should drop fewer items, mainly by picking in careful mode more often. Rules always pick in normal mode.


In [ ]:
# Mean drops, careful picks and score per shift, normal vs raised damage weight
behaviour = (
    df[df['condition'].isin(['familiar', 'high_damage'])]
    .groupby(['policy', 'condition'])[['drops', 'careful', 'score']]
    .mean()
    .round(2)
    .unstack('condition')
    .reindex(ORDER)
)
behaviour


## Results

- Key observations:
- Surprises or failure modes:
- Decision: continue, pivot, or stop:

## Next steps

- CP-SAT clairvoyant bound per shift, to measure how much room remains.
- Hyperparameter sweep: learning rate, gamma, epsilon decay (DQN) and entropy coefficient (PPO), several seeds each.
- Scale training to the Part B budget of 1M steps once the learning curves settle.
